# ***Data Analysis and Dataset Preparation***

### PyshioChallenge 2012 has 3 datasets(a,b and c) in .txt(csv) format with 4 thousand entries each one.(Each patient have your own file)
- *"The data used for the challenge consist of records from 12,000 ICU stays. All patients were adults who were admitted for a wide variety of reasons to cardiac, medical, surgical, and trauma ICUs. ICU stays of less than 48 hours have been excluded. Patients with DNR (do not resuscitate) or CMO (comfort measures only) directives were not excluded."* ¹
- *"All valid values for general descriptors, time series variables, and outcome-related descriptors are non-negative (≥ 0). A value of -1 indicates missing or unknown data (for example, if a patient's height was not recorded)."* ¹
https://physionet.org/content/challenge-2012/1.0.0/

### In this notebook:
- Preprocess patient files and consolidate the data (selecting features and instances) into a single wide-format dataset for training an LSTM model.
- Split the dataset into training and test sets (80%/20%) using stratification.
- Provide metadata information and exploratory data analysis.

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from pathlib import Path
import os
import glob

BASE_DIR = Path().resolve()

### **General Descriptors**


#### Six descriptors were collected at the time the patient is admitted to the ICU.<br> Their associated time-stamps are set to 00:00 (thus they appear at the beginning of each patient's record):

- **RecordID (a unique integer for each ICU stay)**<br>
- **Age (years)**<br>
- **Gender (0: female, or 1: male)**<br>
- **Height (cm)**<br>
- **ICUType (1: Coronary Care Unit, 2: Cardiac Surgery Recovery Unit,3: Medical ICU, or 4: Surgical ICU)**<br>
- **Weight (kg)*.**<br>




#### **Patient File** (Example)

In [2]:
ds1 = pd.read_csv(BASE_DIR / "ds" / "set-a" / "132539.txt")
ds1[ds1['Time'] == '00:00']

,Time,Parameter,Value
0,00:00,RecordID,132539.0
1,00:00,Age,54.0
2,00:00,Gender,0.0
3,00:00,Height,-1.0
4,00:00,ICUType,4.0
5,00:00,Weight,-1.0


### **Time Series**
#### - These 37 variables may be observed once, more than once, or not at all in some cases.

In [3]:
ds1['Parameter'].unique()

<ArrowStringArray>
[  'RecordID',        'Age',     'Gender',     'Height',    'ICUType',
     'Weight',        'GCS',         'HR',  'NIDiasABP',      'NIMAP',
   'NISysABP',   'RespRate',       'Temp',      'Urine',        'HCT',
        'BUN', 'Creatinine',    'Glucose',       'HCO3',         'Mg',
  'Platelets',          'K',         'Na',        'WBC']
Length: 24, dtype: str

| Feature | Description | Unit / Range |
|---|---|---|
| Albumin | Serum albumin | g/dL |
| ALP | Alkaline phosphatase | IU/L |
| ALT | Alanine transaminase | IU/L |
| AST | Aspartate transaminase | IU/L |
| Bilirubin | Serum bilirubin | mg/dL |
| BUN | Blood urea nitrogen | mg/dL |
| Cholesterol | Serum cholesterol | mg/dL |
| Creatinine | Serum creatinine | mg/dL |
| DiasABP | Invasive diastolic arterial blood pressure | mmHg |
| FiO2 | Fractional inspired O₂ | 0–1 |
| GCS | Glasgow Coma Score | 3–15 |
| Glucose | Serum glucose | mg/dL |
| HCO3 | Serum bicarbonate | mmol/L |
| HCT | Hematocrit | % |
| HR | Heart rate | bpm |
| K | Serum potassium | mEq/L |
| Lactate | Serum lactate | mmol/L |
| Mg | Serum magnesium | mmol/L |
| MAP | Invasive mean arterial blood pressure | mmHg |
| MechVent | Mechanical ventilation respiration | 0: false / 1: true |
| Na | Serum sodium | mEq/L |
| NIDiasABP | Non-invasive diastolic arterial blood pressure | mmHg |
| NIMAP | Non-invasive mean arterial blood pressure | mmHg |
| NISysABP | Non-invasive systolic arterial blood pressure | mmHg |
| PaCO2 | Partial pressure of arterial CO₂ | mmHg |
| PaO2 | Partial pressure of arterial O₂ | mmHg |
| pH | Arterial pH | 0–14 |
| Platelets | Platelet count | cells/nL |
| RespRate | Respiration rate | bpm |
| SaO2 | O₂ saturation in hemoglobin | % |
| SysABP | Invasive systolic arterial blood pressure | mmHg |
| Temp | Temperature | °C |
| TropI | Troponin-I | μg/L |
| TropT | Troponin-T | μg/L |
| Urine | Urine output | mL |
| WBC | White blood cell count | cells/nL |
| Weight | Body weight | kg |



#### *Note:* 
##### - Weight is a general descriptor (recorded on admission) AND a time series variable (often measured hourly, for estimating fluid balance).

## ***Outcomes***

### Separate CSV text file
##### 
     - RecordID 
     - SAPS-I score (Le Gall et al., 1984)
     - SOFA score (Ferreira et al., 2001)
     - Length of stay (days)
     - Survival (days)
     - In-hospital death (0: survivor, or 1: died in-hospital)


**If the patient's death was recorded (in or out of hospital), then Survival is the number of days between ICU admission and death; otherwise, Survival is assigned the value -1. Since patients who spent less than 48 hours in the ICU have been excluded, Length of stay and Survival never have the values 0 or 1 in the challenge data sets**

Survival > Length of stay  ⇒  Survivor
Survival = -1  ⇒  Survivor
2 ≤ Survival ≤ Length of stay  ⇒  In-hospital death



## **Preprocessing Datasetets** 

#### Run next cell only one time to merge all patients files data creating a all patients DS; (for a, b and c ds)

In [9]:
###  Creating combined and formated dataset from 3 dataset ###

# Time Conversion: converted the HH:MM format into total Minutes
# replaces −1 with NaN


# Configuration
path = str(BASE_DIR / "ds" / "set-c" / "*.txt")
output_dir = str(BASE_DIR / "ds")
output_file = os.path.join(output_dir, "dsc.csv")

def parse_physionet_file(file_path):
    # Extract Patient ID from filename
    patient_id = os.path.basename(file_path).split('.')[0]
    
    # Read the file
    # Format: Time, Parameter, Value
    df = pd.read_csv(file_path)
    
    # Convert Time (HH:MM) to total minutes or hours for numerical analysis
    # This makes it much easier for an LSTM to process sequences
    def time_to_minutes(t):
        if t == '00:00': return 0
        h, m = map(int, t.split(':'))
        return h * 60 + m

    df['Minutes'] = df['Time'].apply(time_to_minutes)
    df['RecordID'] = patient_id
    
    return df

# Get all file paths
all_files = glob.glob(path)

print(f"Found {len(all_files)} files. Starting processing...")

# List to hold individual dataframes
data_list = []

for i, filename in enumerate(all_files):
    data_list.append(parse_physionet_file(filename))
    
    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1} files...")

# Combine all into one large "Long-Format" DataFrame

final_df = pd.concat(data_list, ignore_index=True)

# Replace missing values (-1) with NaN for better handling in Pandas/Scikit-learn
final_df['Value'] = final_df['Value'].replace(-1, float('nan'))

# Reorder columns for readability
final_df = final_df[['RecordID', 'Minutes', 'Parameter', 'Value']]

# Save to CSV
final_df.to_csv(output_file, index=False)
print(f"Combined data saved to {output_file}")

Found 4000 files. Starting processing...
Processed 500 files...
Processed 1000 files...
Processed 1500 files...
Processed 2000 files...
Processed 2500 files...
Processed 3000 files...
Processed 3500 files...
Processed 4000 files...
Combined data saved to /Users/brunorocha/Aulas/lstmBayesProject/ds/dsc.csv


### **Merge of datasets a,b and c**

In [10]:
#Merge each dataset with your outcomes and then merging all.
dsa = pd.read_csv(BASE_DIR / "ds" / "dsa.csv")
targeta = pd.read_csv(BASE_DIR / "ds" / "outcomes" / "Outcomes-a.txt")
dsa = pd.merge(dsa, targeta, on='RecordID', how='inner')

# --- Conjunto B ---
dsb = pd.read_csv(BASE_DIR / "ds" / "dsb.csv")
targetb = pd.read_csv(BASE_DIR / "ds" / "outcomes" / "Outcomes-b.txt")
dsb = pd.merge(dsb, targetb, on='RecordID', how='inner')

# --- Conjunto C ---
dsc = pd.read_csv(BASE_DIR / "ds" / "dsc.csv")
targetc = pd.read_csv(BASE_DIR / "ds" / "outcomes" / "Outcomes-c.txt")
dsc = pd.merge(dsc, targetc, on='RecordID', how='inner')

ds = pd.concat([dsa, dsb,dsc], axis=0, ignore_index=True)

ds = ds[ds['Parameter'] != 'RecordID']
ds['RecordID'].nunique() # 12000 in total





12000

In [11]:
dsf = ds.pivot_table(index=['RecordID', 'Minutes','SAPS-I', 'SOFA',
       'Length_of_stay', 'Survival', 'In-hospital_death'], 
                       columns='Parameter', 
                       values='Value').reset_index()

In [12]:
dsf.head()

Parameter,RecordID,Minutes,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death,ALP,ALT,AST,...,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,Weight,pH
0,132539,0,6,1,5,-1,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,132539,7,6,1,5,-1,0,NaN,NaN,NaN,...,19.0,NaN,NaN,35.1,NaN,NaN,900.0,NaN,NaN,NaN
2,132539,37,6,1,5,-1,0,NaN,NaN,NaN,...,19.0,NaN,NaN,35.6,NaN,NaN,60.0,NaN,NaN,NaN
3,132539,97,6,1,5,-1,0,NaN,NaN,NaN,...,18.0,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN
4,132539,157,6,1,5,-1,0,NaN,NaN,NaN,...,19.0,NaN,NaN,NaN,NaN,NaN,170.0,NaN,NaN,NaN


In [13]:
dsf.describe()

Parameter,RecordID,Minutes,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death,ALP,ALT,AST,...,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,Weight,pH
count,898352.000000,898352.000000,898352.000000,898352.000000,898352.000000,898352.000000,898352.000000,9435.000000,9705.000000,9706.000000,...,165424.000000,23983.000000,440215.000000,258932.000000,1175.000000,6251.000000,406218.000000,38880.000000,385939.000000,72207.000000
mean,147757.956211,1297.085044,15.430289,7.208419,14.051877,132.326644,0.153647,120.103445,362.829758,505.216979,...,19.631033,96.660671,118.870696,37.032400,8.210383,1.174183,119.372285,13.099816,83.267114,7.420649
std,8786.198866,853.096374,5.800682,4.291584,13.151957,373.571993,0.360610,175.172781,1133.038907,1649.546241,...,5.511886,3.647788,25.163413,1.365156,10.973641,2.785938,164.727363,33.371406,24.525926,4.858785
min,132539.000000,0.000000,-1.000000,-1.000000,-1.000000,-23.000000,0.000000,8.000000,1.000000,4.000000,...,0.000000,0.000000,0.000000,-17.800000,0.100000,0.010000,0.000000,0.000000,0.000000,0.000000
25%,140106.000000,523.000000,12.000000,4.000000,6.000000,-1.000000,0.000000,59.000000,20.000000,30.000000,...,16.000000,96.000000,102.000000,36.600000,0.900000,0.060000,40.000000,8.300000,67.000000,7.330000
50%,147813.000000,1234.000000,16.000000,7.000000,10.000000,-1.000000,0.000000,82.000000,42.000000,62.000000,...,19.000000,97.000000,116.000000,37.100000,3.300000,0.180000,70.000000,11.500000,80.000000,7.380000
75%,155330.000000,2029.000000,19.000000,10.000000,17.000000,18.000000,0.000000,123.500000,146.000000,208.000000,...,23.000000,98.000000,134.000000,37.600000,11.050000,0.895000,140.000000,15.500000,95.000000,7.430000
max,163037.000000,2880.000000,38.000000,22.000000,295.000000,2664.000000,1.000000,4695.000000,16920.000000,36400.000000,...,100.000000,100.000000,295.000000,42.200000,49.600000,29.910000,11000.000000,6251.900000,472.000000,735.000000


In [14]:
#dsf.info() 898352 entries and 48 columns

### Preparing Time Series Format (One-Hour Intervals and Keeping Only the First 12 Hours)

In [15]:

#Transform features for hours (taking the mean if there are more then one value in that hour)
# Temporal features (> 100.000 non-null )
clinical_features = ['Weight', 'GCS', 'HR', 'NIDiasABP', 'NIMAP', 
                      'NISysABP', 'RespRate', 'Temp', 'DiasABP', 'MAP', 'SysABP', 'Urine']
target = 'In-hospital_death'

#  Create the temporal variable 't' (60-minute; capped at t=47)
dsf['t'] = (dsf['Minutes'] // 60).clip(upper=47)

# Group and calculate the mean (Pandas automatically keeps RecordID and t in the index)
grouped_df = dsf.groupby(['RecordID', 't'])[clinical_features].mean()

# Build  temporal grid (All Patients x 48 timesteps)
complete_grid = pd.MultiIndex.from_product(
    [dsf['RecordID'].unique(), range(48)], 
    names=['RecordID', 't']
)

# Reindex to the complete grid and finalize
dsf_complete = grouped_df.reindex(complete_grid).reset_index()
target_df = (
    dsf[['RecordID', target]]
    .drop_duplicates(subset='RecordID')
)

dsf_complete = dsf_complete.merge(target_df, on='RecordID', how='left')

expected_rows = len(dsf['RecordID'].unique()) * 48
print(f"Expected final shape: {expected_rows} rows.")
print(f"Actual final shape: {len(dsf_complete)} rows.")

Expected final shape: 576000 rows.
Actual final shape: 576000 rows.


In [16]:
dsf_complete.info()

<class 'pandas.DataFrame'>
RangeIndex: 576000 entries, 0 to 575999
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RecordID           576000 non-null  int64  
 1   t                  576000 non-null  int64  
 2   Weight             303161 non-null  float64
 3   GCS                184479 non-null  float64
 4   HR                 519023 non-null  float64
 5   NIDiasABP          241952 non-null  float64
 6   NIMAP              238749 non-null  float64
 7   NISysABP           242298 non-null  float64
 8   RespRate           138445 non-null  float64
 9   Temp               213670 non-null  float64
 10  DiasABP            311795 non-null  float64
 11  MAP                310021 non-null  float64
 12  SysABP             311823 non-null  float64
 13  Urine              398612 non-null  float64
 14  In-hospital_death  576000 non-null  int64  
dtypes: float64(12), int64(3)
memory usage: 65.9 MB


In [17]:
#checking dataset with categorical time in hours
stats_complete = dsf_complete.describe().T[['mean', 'std']].rename(
    columns={'mean': 'Média_Complete', 'std': 'DP_Complete'}
)

stats_af = dsf.describe().T[['mean', 'std']].rename(
    columns={'mean': 'Média_Af', 'std': 'DP_Af'}
)

comp = pd.concat([stats_complete, stats_af], axis=1)

comp['Dif_Media'] = (comp['Média_Complete'] - comp['Média_Af']).abs()

print(comp['Dif_Media'].dropna())

Parameter
RecordID             13.261461
t                     2.364971
Weight                0.075630
GCS                   0.007462
HR                    0.786930
NIDiasABP             0.227854
NIMAP                 0.346589
NISysABP              0.666919
RespRate              0.037903
Temp                  0.043957
DiasABP               0.289591
MAP                   0.546297
SysABP                1.042814
Urine                 2.436766
In-hospital_death     0.011397
Name: Dif_Media, dtype: float64


In [18]:
#  Keep only the first 12 hours (t = 0 to t = 11)
dsf_filtered = dsf_complete[dsf_complete['t'] <= 11].copy()

## **Missing Data Analysis** (for instances and features selection)

In [19]:
n_rows, n_cols = dsf_filtered.shape

na_col_abs = dsf_filtered.isna().sum()
na_col_pct = dsf_filtered.isna().mean() * 100

na_col_summary = pd.DataFrame({
    
    'NA_abs': na_col_abs,
    'NA_pct': na_col_pct
}).sort_values(by='NA_pct', ascending=False)

na_col_summary.head(48)

,NA_abs,NA_pct
Parameter,,
RespRate,109715,76.190972
GCS,94356,65.525000
Temp,87857,61.011806
Weight,80303,55.765972
NIMAP,79029,54.881250
NIDiasABP,78273,54.356250
NISysABP,78191,54.299306
MAP,75969,52.756250
DiasABP,75243,52.252083


In [20]:
dsf_filtered.info()

<class 'pandas.DataFrame'>
Index: 144000 entries, 0 to 575963
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RecordID           144000 non-null  int64  
 1   t                  144000 non-null  int64  
 2   Weight             63697 non-null   float64
 3   GCS                49644 non-null   float64
 4   HR                 123654 non-null  float64
 5   NIDiasABP          65727 non-null   float64
 6   NIMAP              64971 non-null   float64
 7   NISysABP           65809 non-null   float64
 8   RespRate           34285 non-null   float64
 9   Temp               56143 non-null   float64
 10  DiasABP            68757 non-null   float64
 11  MAP                68031 non-null   float64
 12  SysABP             68767 non-null   float64
 13  Urine              93551 non-null   float64
 14  In-hospital_death  144000 non-null  int64  
dtypes: float64(12), int64(3)
memory usage: 17.6 MB


In [21]:
# Chossing only one measure of ABP 
bp_pairs = [
    ("DiasABP", "NIDiasABP"),
    ("MAP", "NIMAP"),
    ("SysABP", "NISysABP")
]

for invasive, non_invasive in bp_pairs:
    new_col = invasive + "f"
    
    dsf_filtered[new_col] = dsf_filtered[invasive].combine_first(
        dsf_filtered[non_invasive]
    )
cols_to_drop = [col for pair in bp_pairs for col in pair]
dsf_filtered = dsf_filtered.drop(columns=cols_to_drop)

In [22]:
dsf_filtered.info()

<class 'pandas.DataFrame'>
Index: 144000 entries, 0 to 575963
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RecordID           144000 non-null  int64  
 1   t                  144000 non-null  int64  
 2   Weight             63697 non-null   float64
 3   GCS                49644 non-null   float64
 4   HR                 123654 non-null  float64
 5   RespRate           34285 non-null   float64
 6   Temp               56143 non-null   float64
 7   Urine              93551 non-null   float64
 8   In-hospital_death  144000 non-null  int64  
 9   DiasABPf           122482 non-null  float64
 10  MAPf               121476 non-null  float64
 11  SysABPf            122506 non-null  float64
dtypes: float64(9), int64(3)
memory usage: 14.3 MB


In [23]:
# Cheacking problematic instances
    
def checking_Nas(dsf_filtered):
    features = [col for col in dsf_filtered.columns if col not in ['RecordID', 't','In-hospital_death']]

    bad_ids_all = set()
    bad_ids_summary = {}

    for feature in features:
        counts = dsf_filtered.groupby('RecordID')[feature].count()
        
        
        if feature in ['GCS', 'Urine', 'Temp']:
            threshold = 3
        else:
            threshold = 6
        
        bad_ids = counts[counts < threshold].index
        
        
        bad_ids_summary[feature] = len(bad_ids)
        
        
        bad_ids_all.update(bad_ids)
    return bad_ids_summary,bad_ids_all






In [24]:

print("\n Number of problematic instances per feature:")
bad_ids_summary,bad_ids_all =checking_Nas(dsf_filtered)
for feat, count in sorted(bad_ids_summary.items(), key=lambda x: -x[1]):
    print(f"{feat}: {count}")

print(f"\n Total unique problematic instances: {len(bad_ids_all)}")



 Number of problematic instances per feature:
RespRate: 8811
Weight: 6748
GCS: 1989
Temp: 1787
Urine: 1147
MAPf: 504
DiasABPf: 454
SysABPf: 452
HR: 442

 Total unique problematic instances: 10926


In [25]:
# drop features ['RespRate', 'Weight']
dsf_filtered = dsf_filtered.drop(columns=['RespRate', 'Weight'])

In [26]:
#checking drop
dsf_filtered.columns

Index(['RecordID', 't', 'GCS', 'HR', 'Temp', 'Urine', 'In-hospital_death',
       'DiasABPf', 'MAPf', 'SysABPf'],
      dtype='str', name='Parameter')

In [27]:
# cheking without features 'RespRate', 'Weight'
print("\n Number of problematic instances per feature:")
bad_ids_summary,bad_ids_all =checking_Nas(dsf_filtered)
for feat, count in sorted(bad_ids_summary.items(), key=lambda x: -x[1]):
    print(f"{feat}: {count}")

print(f"\n Total unique bad_ids: {len(bad_ids_all)}")



 Number of problematic instances per feature:
GCS: 1989
Temp: 1787
Urine: 1147
MAPf: 504
DiasABPf: 454
SysABPf: 452
HR: 442

 Total unique bad_ids: 3584


In [28]:
#deleting bad_IDs;
# Similar Proportions after delete;
print(dsf_filtered['In-hospital_death'].value_counts(normalize=True))
dsf_filtered = dsf_filtered[
    ~dsf_filtered['RecordID'].isin(bad_ids_all)
]
print(dsf_filtered['In-hospital_death'].value_counts(normalize=True))


In-hospital_death
0    0.85775
1    0.14225
Name: proportion, dtype: float64
In-hospital_death
0    0.854206
1    0.145794
Name: proportion, dtype: float64


In [29]:
dsf_filtered.info()

<class 'pandas.DataFrame'>
Index: 100992 entries, 0 to 575963
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RecordID           100992 non-null  int64  
 1   t                  100992 non-null  int64  
 2   GCS                39728 non-null   float64
 3   HR                 91039 non-null   float64
 4   Temp               44210 non-null   float64
 5   Urine              74688 non-null   float64
 6   In-hospital_death  100992 non-null  int64  
 7   DiasABPf           90403 non-null   float64
 8   MAPf               89853 non-null   float64
 9   SysABPf            90415 non-null   float64
dtypes: float64(7), int64(3)
memory usage: 8.5 MB


In [30]:

temp_features = [col for col in dsf_filtered.columns if col not in ['RecordID', 't','In-hospital_death']]
dsf_wide = dsf_filtered.pivot(index='RecordID', columns='t', values=temp_features)
#Flatten the MultiIndex columns into a single string
# This renames the columns from a tuple ('Weight', 0) to 'Weight_t0'
dsf_wide.columns = [f"{feature}_t{t}" for feature, t in dsf_wide.columns]
# Reset the index so 'RecordID' becomes a standard column again
dsf_wide = dsf_wide.reset_index()
target_df = (
    dsf_filtered[['RecordID', target]]
    .drop_duplicates(subset='RecordID')
)

dsf_wide = dsf_wide.merge(target_df, on='RecordID', how='left')

# Check the new shape and columns
print(f"Number of patients: {len(dsf_wide)}")
print(f"Number of columns: {len(dsf_wide.columns)} (1 RecordID + 1 target + 7 features * 12 timesteps)")
print(dsf_wide.head())

Number of patients: 8416
Number of columns: 86 (1 RecordID + 1 target + 7 features * 12 timesteps)
   RecordID  GCS_t0  GCS_t1  GCS_t2  GCS_t3  GCS_t4  GCS_t5  GCS_t6  GCS_t7  \
0    132539    15.0     NaN     NaN    15.0     NaN     NaN     NaN    15.0   
1    132540     NaN     3.0     NaN    10.0    11.0     NaN     NaN    15.0   
2    132541     7.0     NaN     8.0     NaN     NaN     NaN     8.0     NaN   
3    132545     NaN     NaN    15.0     NaN    15.0     NaN     NaN     NaN   
4    132548    15.0    15.0     NaN     NaN     NaN    15.0     NaN     NaN   

   GCS_t8  ...  SysABPf_t3  SysABPf_t4  SysABPf_t5  SysABPf_t6  SysABPf_t7  \
0     NaN  ...      114.00         NaN       110.0         NaN       107.0   
1     NaN  ...       94.00        99.0        89.0  107.000000       114.0   
2     NaN  ...      125.00       134.0       144.0  138.000000       138.0   
3    15.0  ...      151.00       130.0       119.0  139.333333       151.0   
4     NaN  ...      201.25         N

## **Data Stratification and Splitting**

In [31]:
dsf_wide['In-hospital_death'].value_counts(normalize=True) # 85%/15%

In-hospital_death
0    0.854206
1    0.145794
Name: proportion, dtype: float64

In [32]:
#df.info()
#df.columns

In [33]:


def stratified_train_test_split(df, test_size=0.20, random_state=42):
    """
    Splits a wide-format clinical dataset into train and test sets.
    Stratifies by 'In-hospital_death' and the approximate number of 
    non-NA values for GCS, Urine, and Temp across t0 to t11.
    recieve df(wide) and return train_df(wide), test_df(wide)
    """
    # Create a copy to avoid SettingWithCopy warnings if df is a slice
    df_processed = df.copy()
    
    # 1. Define the column groups
    gcs_cols = [f'GCS_t{i}' for i in range(12)]
    urine_cols = [f'Urine_t{i}' for i in range(12)]
    temp_cols = [f'Temp_t{i}' for i in range(12)]
    
    # 2. Count non-null values for each feature group per row (max = 12)
    df_processed['GCS_count'] = df_processed[gcs_cols].notna().sum(axis=1)
    df_processed['Urine_count'] = df_processed[urine_cols].notna().sum(axis=1)
    df_processed['Temp_count'] = df_processed[temp_cols].notna().sum(axis=1)
    
    # 3. Categorize the counts: < 6 is 'Low', >= 6 is 'Normal'
    def categorize_count(count):
        return 'Low' if count < 6 else 'Normal'
        
    df_processed['GCS_bin'] = df_processed['GCS_count'].apply(categorize_count)
    df_processed['Urine_bin'] = df_processed['Urine_count'].apply(categorize_count)
    df_processed['Temp_bin'] = df_processed['Temp_count'].apply(categorize_count)
    
    # 4. Create a combined stratification key
    # Example output: "1_Low_Normal_Low"
    df_processed['stratify_key'] = (
        df_processed['In-hospital_death'].astype(str) + "_" +
        df_processed['GCS_bin'] + "_" +
        df_processed['Urine_bin'] + "_" +
        df_processed['Temp_bin']
    )
    
    # Handle rare combinations properly
    # Find combinations that appear less than 2 times (which break the split)
    key_counts = df_processed['stratify_key'].value_counts()
    rare_keys = key_counts[key_counts < 2].index
    
    # If there are any rare keys, merge them into the MOST frequent class
    if len(rare_keys) > 0:
        most_frequent_key = key_counts.idxmax()
        df_processed.loc[df_processed['stratify_key'].isin(rare_keys), 'stratify_key'] = most_frequent_key
    
    
    # 6. Perform the stratified split
    train_df, test_df = train_test_split(
        df_processed, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=df_processed['stratify_key']
    )
    #Print the counts of 'Low' for each feature
    # print("--- 'Low' counts (< 6 non-NAs) in Train and Test ---")
    # for feature in ['GCS', 'Urine', 'Temp']:
    #     bin_col = f"{feature}_bin"
        
    #     train_low_count = (train_df[bin_col] == 'Low').sum()
    #     test_low_count = (test_df[bin_col] == 'Low').sum()
        
    #     # Calcular a percentagem para verificar se a estratificação ficou equilibrada
    #     train_pct = (train_low_count / len(train_df)) * 100
    #     test_pct = (test_low_count / len(test_df)) * 100
        
    #     print(f"{feature:>5}: Train = {train_low_count} ({train_pct:.1f}%) | Test = {test_low_count} ({test_pct:.1f}%)")
    # print("----------------------------------------------------\n")
    
    # 7. Clean up the temporary helper columns to restore the original dataframe structure
    cols_to_drop = [
        'GCS_count', 'Urine_count', 'Temp_count', 
        'GCS_bin', 'Urine_bin', 'Temp_bin', 'stratify_key'
    ]
    
    train_df = train_df.drop(columns=cols_to_drop)
    
    
    return train_df, test_df



In [35]:
train_data, test_data = stratified_train_test_split(dsf_wide)
print(f"Train set size: {train_data.shape}")
print(f"Test set size: {test_data.shape}")

Train set size: (6732, 86)
Test set size: (1684, 93)


In [36]:
print(train_data['In-hospital_death'].value_counts(normalize = True))
print(test_data['In-hospital_death'].value_counts(normalize = True ))

In-hospital_death
0    0.854278
1    0.145722
Name: proportion, dtype: float64
In-hospital_death
0    0.853919
1    0.146081
Name: proportion, dtype: float64


In [37]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 6732 entries, 2217 to 7191
Data columns (total 86 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RecordID           6732 non-null   int64  
 1   GCS_t0             2675 non-null   float64
 2   GCS_t1             2614 non-null   float64
 3   GCS_t2             2593 non-null   float64
 4   GCS_t3             2675 non-null   float64
 5   GCS_t4             2696 non-null   float64
 6   GCS_t5             2793 non-null   float64
 7   GCS_t6             2743 non-null   float64
 8   GCS_t7             2629 non-null   float64
 9   GCS_t8             2535 non-null   float64
 10  GCS_t9             2653 non-null   float64
 11  GCS_t10            2613 non-null   float64
 12  GCS_t11            2569 non-null   float64
 13  HR_t0              3547 non-null   float64
 14  HR_t1              5177 non-null   float64
 15  HR_t2              5927 non-null   float64
 16  HR_t3              6273 non-null   fl

In [38]:
test_data.info()

<class 'pandas.DataFrame'>
Index: 1684 entries, 5227 to 5338
Data columns (total 93 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RecordID           1684 non-null   int64  
 1   GCS_t0             671 non-null    float64
 2   GCS_t1             668 non-null    float64
 3   GCS_t2             622 non-null    float64
 4   GCS_t3             683 non-null    float64
 5   GCS_t4             649 non-null    float64
 6   GCS_t5             725 non-null    float64
 7   GCS_t6             677 non-null    float64
 8   GCS_t7             662 non-null    float64
 9   GCS_t8             592 non-null    float64
 10  GCS_t9             690 non-null    float64
 11  GCS_t10            684 non-null    float64
 12  GCS_t11            617 non-null    float64
 13  HR_t0              892 non-null    float64
 14  HR_t1              1309 non-null   float64
 15  HR_t2              1500 non-null   float64
 16  HR_t3              1559 non-null   fl

In [39]:
#saving the dataset with NA

train_data.to_csv("/Users/brunorocha/Aulas/IACD/Sem6/Projeto/lstmBayesProject/ds_lstm/trainLSTM.csv",index=False)

test_data.to_csv("/Users/brunorocha/Aulas/IACD/Sem6/Projeto/lstmBayesProject/ds_lstm/testLSTM.csv",index=False)


## **Reference**
1. https://physionet.org/content/challenge-2012/1.0.0/#files-panel